In [ ]:
from datasets import load_dataset

dataset = load_dataset("daekeun-ml/naver-news-summarization-ko")

In [ ]:
data = dataset
data

In [ ]:
data["train"]["document"][0]

In [ ]:
print(sorted(list(set(data["train"]["document"][0]))))

In [ ]:
# 이것은 텍스트에 나타나는 모든 문자 집합을 가져온 것입니다.
# 그것들은 고유한 것들만 남겨서 sorted로 정렬을 하면 아래와 같은 결과를 볼 수 있습니다.
ko_text = "".join(data["train"]["document"])
ko_chars = sorted(list(set((ko_text))))
ko_vocab_size = len(ko_chars)
print("총 글자 수 :", ko_vocab_size)

In [ ]:
character_to_ids = {char: i for i, char in enumerate(ko_chars)}
ids_to_character = {i: char for i, char in enumerate(ko_chars)}

In [ ]:
token_encode = lambda s: [character_to_ids[c] for c in s]
token_decode = lambda l: "".join([ids_to_character[i] for i in l])

In [ ]:
print(token_encode("안녕하세요 함께 인공지능을 공부하게 되어 반가워요."))
print(token_decode(token_encode("안녕하세요 함께 인공지능을 공부하게 되어 반가워요.")))

In [ ]:
import torch

tokenized_data = torch.tensor(token_encode(ko_text), dtype=torch.long)
print(tokenized_data.shape, tokenized_data.dtype)
print(tokenized_data[:100])

In [ ]:
n = int(0.9 * len(tokenized_data))
train_dataset = tokenized_data[:n]
test_dataset = tokenized_data[n:]

In [ ]:
block_size = 8
train_dataset[:block_size]

In [ ]:
x = train_dataset[:block_size]
y = train_dataset[1:block_size + 1]

for time in range(block_size):
    context = x[:time + 1]
    target = y[time]

    print(f"입력 텐셔 : {context}")
    print(f"타켓 글자 : {target}")

In [ ]:
torch.manual_seed(1234)

batch_size = 4
block_size = 8


def batch_function(mode):
    dataset = train_dataset if mode == "train" else test_dataset
    idx = torch.randint(len(dataset) - block_size, (batch_size,))
    x = torch.stack([dataset[index:index + block_size] for index in idx])
    y = torch.stack([dataset[index + 1:index + block_size + 1] for index in idx])
    return x, y


example_x, example_y = batch_function("train")
print("inputs : ", example_x.shape)
print("")
print("example_x의 실제 값")
print(example_x)
print("-----------------------")
print("targets : ", example_y.shape)
print("")
print("example_y의 실제 값")
print(example_y)
print("-----------------------")

for size in range(batch_size):
    for t in range(block_size):
        context = example_x[size, :t + 1]
        target = example_y[size, t]
        print(f"input : {context}, target : {target}")
    print("-----------------------")
    print("-----------------------")

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F


class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.embedding_token_table = nn.Embedding(vocab_length, vocab_length)

    def forward(self, inputs, targets):
        logits = self.embedding_token_table(inputs)

        return logits


model = semiGPT(ko_vocab_size)
output = model(example_x, example_y)
print(output.shape)

In [ ]:
#에러가 발생되도록 세팅된 코드
import torch
import torch.nn as nn
from torch.nn import functional as F


class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.embedding_token_table = nn.Embedding(vocab_length, vocab_length)

    def forward(self, inputs, targets):
        logits = self.embedding_token_table(inputs)

        loss = F.cross_entropy(logits, targets)
        return logits, loss


model = semiGPT(ko_vocab_size)
output, loss = model(example_x, example_y)
print(output)

In [17]:
import torch
import torch.nn as nn
from torch.nn import functional as F


class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.embedding_token_table = nn.Embedding(vocab_length, vocab_length)

    def forward(self, inputs, targets):
        logits = self.embedding_token_table(inputs)
        batch, seq_length, vocab_length = logits.shape
        logits = logits.view(batch * seq_length, vocab_length)
        targets = targets.view(batch * seq_length)
        loss = F.cross_entropy(logits, targets)

        print("logits의 shape는 : ", logits.shape, "입니다.")
        print("targets의 shape는 : ", targets.shape, "입니다.")

        return logits, loss


model = semiGPT(ko_vocab_size)
logits, loss = model(example_x, example_y)
print(loss)

logits의 shape는 :  torch.Size([32, 2701]) 입니다.
targets의 shape는 :  torch.Size([32]) 입니다.
tensor(8.5369, grad_fn=<NllLossBackward0>)


In [18]:
example_x.shape, example_y.shape

(torch.Size([4, 8]), torch.Size([4, 8]))

In [19]:
import torch
import torch.nn as nn
from torch.nn import functional as F


class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.embedding_token_table = nn.Embedding(vocab_length, vocab_length)

    def forward(self, inputs, targets=None):
        logits = self.embedding_token_table(inputs)
        if targets is None:
            loss = None
        else:
            batch, seq_length, vocab_length = logits.shape
            logits = logits.view(batch * seq_length, vocab_length)
            targets = targets.view(batch * seq_length)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, inputs, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(inputs)
            logits = logits[:, -1, :]
            print(logits.shape)
            probs = F.softmax(logits, dim=-1)
            next_inputs = torch.multinomial(probs, num_samples=1)
            inputs = torch.cat((inputs, next_inputs), dim=1)
        return inputs


model = semiGPT(ko_vocab_size)
logits, loss = model(example_x, example_y)
print(loss)

token_decode(model.generate(torch.zeros((1, 1),
                                        dtype=torch.long),
                            max_new_tokens=10)[0].tolist())

tensor(8.5844, grad_fn=<NllLossBackward0>)
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])


' 쐐『テ토龍t灘÷확절'

In [23]:
import torch

logits = torch.tensor(
    [
        [
            [0.1, 0.2, 0.3, 0.4],
            [0.2, 0.3, 0.4, 0.1],
            [0.3, 0.4, 0.1, 0.2]
        ]
    ]
)

result = logits[:, -1, :]
print("선택되는 값        : ", result)
print("결과에 대한 size 값 : ", result.size())

선택되는 값        :  tensor([[0.3000, 0.4000, 0.1000, 0.2000]])
결과에 대한 size 값 :  torch.Size([1, 4])
